# <center> 2707. Extra Characters in a String </center>


## Problem Description
[Click here](https://leetcode.com/problems/extra-characters-in-a-string/description/)


## Intuition
<!-- Describe your first thoughts on how to solve this problem. -->
We are given a string and a dictionary of valid words. We need to split the string into dictionary words such that the number of unused characters is minimized.

At any string index, we must decide whether the current character belongs to a dictionary word:
  - if not, count it as an extra character and move to the next index to continue with the rest of the string
  - if it starts a dictionary word, it means we found a complete word; use that word and move to the index right after it ends

Since the same suffix can be reached through different choices, use DFS + memoization, where `dfs(i)` represents the minimum number of extra characters in `s[i:]`.

To efficiently find all dictionary words starting at a given index, build a Trie containing all words. <br>
Starting from an index, traverse the Trie character by character. If the path does not exist, stop early because no dictionary word can start with that prefix.

Return the minimum extra characters among all possible choices.


## Approach
<!-- Describe your approach to solving the problem. -->
**Class TrieNode**

**init()**
- set children = a hashmap to store the child nodes
    - *key = characters*
    - *value = child nodes*
- set isEnd = false to track if the current node is the last node (character) of a word i.e end of a word

**Class Solution**

**minExtraChar()**
- store all dictionary words in a trie
    - create a root node
    - for each word in the dictionary
        - set cur = root node
        - for each character in the word
            - if the character is not a child node of the current node
              - add it as a child node
            - move to the child node
        - at the end of the loop, mark the current node as the last node
- set memo = a hashmap to cache previously computed results
  - *key = starting index in s*
  - *value = minimum extra characters from that index to the end of the string*
- define a helper function for DFS + memoization <br>
*dfs(current index) → returns the minimum number of extra characters needed for the substring starting at index*
  - if index reaches the end of the string
    - return 0
  - if the result is already cached i.e in memo
    - return it
  - treat the current character as unused (an extra character)
      - set result = 1 + dfs(next index) to recursively find the minimum extra characters in the remaining substring
        - *add 1 for counting the current character as an extra character*
  - set current node = root to start searching for dictionary words beginning at the current index
  - starting from the current index, traverse the trie
    *for each position j from index to the end of the string*
      - if the current character does not exist in the trie i.e s[j] is not a child of the current trie node
        - break the loop to stop searching
      - move to the child node
      - if the current node is the end of a word, a valid word has been found from index to j
          - update result with min(current result, dfs(next index))
  - cache the result i.e add it to memo
  - return the result
- start dfs from index 0 to get the minimum number of extra characters for the entire string

## Complexity
<!-- Add your time complexity here, e.g. $$O(n)$$ -->
- Time complexity:
  - minExtraChar(): O(building trie + DFS with memoization) → O(total characters in dictionary + total recursive calls * work per call) → O(total words * max-length word + total states explored * trie traversal) → O(x * w + n * w) → O((x + n) * w)
    - *there are n states because dfs(index) is computed once for each index due to memoization*
    - *for each state, trie traversal is bounded by the maximum word length w*
    - *therefore total recursive calls = total states explored*

<!-- Add your space complexity here, e.g. $$O(n)$$ -->
- Space complexity:
  - minExtraChar(): O(trie + memo hashmap + recursion stack) → O(total words * max-length word + total states + recursion depth) → O(x * w + n + n) → O(x * w + n)
    - *trie requires O(x × w) space because a node may be created for each character*
    - *memo hashmap stores one result for each index*
    - *recursion depth is n because we process one character per level*


## Code

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.isEnd = False


class Solution:
    def minExtraChar(self, s: str, dictionary: List[str]) -> int:
        root = TrieNode()
        for word in dictionary:
            node = root
            for char in word:
                if char not in node.children:
                    node.children[char] = TrieNode()
                node = node.children[char]
            node.isEnd = True

        memo = {}

        def dfs(i: int) -> int:
            if i == len(s):
                return 0
            if i in memo:
                return memo[i]
            res = 1 + dfs(i + 1)
            node = root
            for j in range(i, len(s)):
                if s[j] not in node.children:
                    break
                node = node.children[s[j]]
                if node.isEnd:
                    res = min(res, dfs(j + 1))
            memo[i] = res
            return res

        return dfs(0)